# 04 - Transformer Model (distilBERT)

Fine-tune a DistilBERT transformer and compare it against the calibrated
Linear SVM baseline from `03_model_training.ipynb`.

- Same train/calibration/test split recipe (64/16/20) so the comparison is fair
- Uses the **raw** message text: BERT's subword tokenizer handles punctuation
  natively, so the `clean_text()` step is skipped
- Metrics are saved to `results/` next to the baseline's

> **Prerequisites:** `pip install -r requirements-transformers.txt` and the
> datasets in `data/raw/`. A GPU speeds training up a lot, but this runs on
> CPU too. Quick-run overrides: `MAX_SAMPLES`, `NUM_EPOCHS`, `BATCH_SIZE`.

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    brier_score_loss,
    classification_report,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

# Allow importing the project's src modules regardless of where Jupyter was launched
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from preprocessing import load_email_data, load_sms_data  # noqa: E402

%matplotlib inline
print(f"torch {torch.__version__} | device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## 1. Configuration

In [ ]:
# Quick-run overrides for CPU testing (set as environment variables)
MAX_SAMPLES = int(os.environ.get("MAX_SAMPLES", "0")) or None
NUM_EPOCHS = int(os.environ.get("NUM_EPOCHS", "3"))
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "32"))
MAX_LEN = 128
MODEL_NAME = "distilbert-base-uncased"
print(f"epochs={NUM_EPOCHS} batch={BATCH_SIZE} max_len={MAX_LEN}")

## 2. Load and split the data

In [ ]:
sms = load_sms_data()
email = load_email_data()
df = pd.concat([sms, email], ignore_index=True)
df["message"] = df["message"].fillna("").astype(str)

if MAX_SAMPLES:
    df = df.sample(n=MAX_SAMPLES, random_state=42)
    print(f"NOTE: quick-run override — using {len(df):,} samples")

print(f"Total messages: {len(df):,}")

# Same split recipe as notebook 03 (64/16/20) for a fair comparison.
# The calibration split is unused here — it only mirrors the baseline setup.
X_train, X_test, y_train, y_test = train_test_split(
    df["message"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)
X_train, X_cal, y_train, y_cal = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)
print(f"Train: {len(X_train):,} | Calibration: {len(X_cal):,} | Test: {len(X_test):,}")

## 3. Tokenize

DistilBERT uses a WordPiece subword tokenizer; we truncate to 128 tokens.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SpamDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx], truncation=True, max_length=MAX_LEN, return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"][0],
            "attention_mask": enc["attention_mask"][0],
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = SpamDataset(X_train, y_train)
eval_ds = SpamDataset(X_test, y_test)
data_collator = DataCollatorWithPadding(tokenizer)
print(f"Train examples: {len(train_ds):,} | Eval examples: {len(eval_ds):,}")

## 4. Fine-tune

The `Trainer` API handles batching, evaluation, and keeping the best
checkpoint (by F1).

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1_score(labels, preds)}

training_args = TrainingArguments(
    output_dir=str(PROJECT_ROOT / "models" / "transformer_checkpoints"),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()

## 5. Evaluate on the held-out test set

In [ ]:
pred = trainer.predict(eval_ds)
y_pred = np.argmax(pred.predictions, axis=1)
proba = torch.softmax(torch.tensor(pred.predictions), dim=1).numpy()[:, 1]

print(classification_report(y_test, y_pred, target_names=["ham", "spam"]))
print(f"ROC-AUC: {roc_auc_score(y_test, proba):.4f}")
print(f"Brier:   {brier_score_loss(y_test, proba):.4f}")

## 6. Compare with the Linear SVM baseline

Loads `results/metrics_linear_svm.json` produced by notebook 03.

In [ ]:
baseline = json.loads((PROJECT_ROOT / "results" / "metrics_linear_svm.json").read_text())

rows = [
    {
        "model": baseline["model"],
        **{k: baseline[k] for k in ("accuracy", "f1_spam", "roc_auc", "brier")},
    },
    {
        "model": MODEL_NAME,
        "accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
        "f1_spam": round(float(f1_score(y_test, y_pred)), 4),
        "roc_auc": round(float(roc_auc_score(y_test, proba)), 4),
        "brier": round(float(brier_score_loss(y_test, proba)), 4),
    },
]
pd.DataFrame(rows).set_index("model")

## 7. Save model and results

In [ ]:
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)

with open(results_dir / "metrics_distilbert.json", "w") as f:
    json.dump(
        {
            "model": MODEL_NAME,
            "accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
            "f1_spam": round(float(f1_score(y_test, y_pred)), 4),
            "roc_auc": round(float(roc_auc_score(y_test, proba)), 4),
            "brier": round(float(brier_score_loss(y_test, proba)), 4),
            "num_train_epochs": NUM_EPOCHS,
        },
        f,
        indent=2,
    )

fig, ax = plt.subplots()
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["ham", "spam"], cmap="Blues", ax=ax
)
ax.set_title("Confusion matrix — distilBERT")
fig.tight_layout()
fig.savefig(results_dir / "confusion_matrix_distilbert.png", dpi=150)
plt.show()

model_dir = PROJECT_ROOT / "models" / "transformer"
model.save_pretrained(str(model_dir))
tokenizer.save_pretrained(str(model_dir))
print(f"Saved model + tokenizer to {model_dir}")